# Bank Marketing — EDA y Modelado (dataset extendido, 20 variables — BONUS)

Hackathon 6 — predecir si un cliente aceptará un depósito a plazo (`y ∈ {yes, no}`).

Dataset: `bank-additional-full.csv` (UCI Bank Marketing, **20 variables**: 19
features + `y`). El profesor pidió estandarizar en el dataset de 17
variables (`bank-full.csv`, ver `notebooks/modeling_standard.ipynb`); este
notebook es la comparación **opcional** con el dataset extendido (+4 puntos
de participación), que agrega indicadores socio-económicos. Ambos modelos
quedan desplegados en paralelo en Cloud Run.

La lógica reutilizable (features, preprocesamiento, entrenamiento) vive en
`src/` y es la que realmente usa la API — este notebook la ejecuta para
mostrar el proceso y los resultados, no la duplica.


In [1]:
import sys
from pathlib import Path

import pandas as pd

sys.path.append(str(Path("..") / "src"))

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 160)

df = pd.read_csv("../data/bank-additional-full.csv", sep=";")
df.shape

(41188, 21)

## 1. Estructura del dataset (20 variables)

In [2]:
df.dtypes

age                 int64
job                   str
marital               str
education             str
default               str
housing               str
loan                  str
contact               str
month                 str
day_of_week           str
duration            int64
campaign            int64
pdays               int64
previous            int64
poutcome              str
emp.var.rate      float64
cons.price.idx    float64
cons.conf.idx     float64
euribor3m         float64
nr.employed       float64
y                     str
dtype: object

In [3]:
df.head()

,age,job,marital,education,default,housing,loan,contact,month,day_of_week,duration,campaign,pdays,previous,poutcome,emp.var.rate,cons.price.idx,cons.conf.idx,euribor3m,nr.employed,y
0,56,housemaid,married,basic.4y,no,no,no,telephone,may,mon,261,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
1,57,services,married,high.school,unknown,no,no,telephone,may,mon,149,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
2,37,services,married,high.school,no,yes,no,telephone,may,mon,226,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
3,40,admin.,married,basic.6y,no,no,no,telephone,may,mon,151,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
4,56,services,married,high.school,no,no,yes,telephone,may,mon,307,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no


## 2. Variable objetivo y desbalance de clases

In [4]:
print(df["y"].value_counts())
print()
print(df["y"].value_counts(normalize=True))

y
no     36548
yes     4640
Name: count, dtype: int64

y
no     0.887346
yes    0.112654
Name: proportion, dtype: float64


El dataset está fuertemente desbalanceado: **~88.7% "no" vs ~11.3% "yes"**.
Se usa `class_weight="balanced"` en los modelos y se prioriza **F1** y
**Balanced Accuracy**, además de **calibrar el umbral de decisión** (ver
sección 6) en vez de usar el 0.5 por defecto.

## 3. Calidad de datos: nulos, duplicados y 'unknown'

In [5]:
print("Missing values (NaN):", df.isna().sum().sum())
print("Duplicated rows:", df.duplicated().sum())
print()
print("Valores 'unknown' por columna categórica:")
for c in df.select_dtypes(include="object").columns:
    n = (df[c] == "unknown").sum()
    if n > 0:
        print(f"  {c}: {n} ({n/len(df)*100:.1f}%)")

Missing values (NaN): 0
Duplicated rows: 12

Valores 'unknown' por columna categórica:
  job: 330 (0.8%)
  marital: 80 (0.2%)
  education: 1731 (4.2%)
  default: 8597 (20.9%)
  housing: 990 (2.4%)
  loan: 990 (2.4%)


C:\Users\bihondaepiquien\AppData\Local\Temp\ipykernel_8612\922973187.py:5: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  for c in df.select_dtypes(include="object").columns:


No hay valores `NaN` explícitos, pero varias columnas categóricas codifican
los faltantes como la categoría `"unknown"` (hasta 20.9% en `default`). Se
mantienen como una categoría más (el One-Hot Encoding la captura de forma
natural). Hay 12 filas duplicadas exactas, que se eliminan antes de
entrenar.

## 4. Variables numéricas y categóricas

In [6]:
from preprocessing import EXTENDED_NUMERIC_FEATURES, EXTENDED_CATEGORICAL_FEATURES, EXTENDED_FEATURES, LEAKAGE_FEATURES

print("Numéricas:", EXTENDED_NUMERIC_FEATURES)
print()
print("Categóricas:", EXTENDED_CATEGORICAL_FEATURES)
print()
print("Excluidas por leakage:", LEAKAGE_FEATURES)
print()
print("Total features finales:", len(EXTENDED_FEATURES))

Numéricas: ['age', 'campaign', 'pdays', 'previous', 'emp.var.rate', 'cons.price.idx', 'cons.conf.idx', 'euribor3m', 'nr.employed']

Categóricas: ['job', 'marital', 'education', 'default', 'housing', 'loan', 'contact', 'month', 'day_of_week', 'poutcome']

Excluidas por leakage: ['duration']

Total features finales: 19


In [7]:
df[EXTENDED_NUMERIC_FEATURES].describe()

,age,campaign,pdays,previous,emp.var.rate,cons.price.idx,cons.conf.idx,euribor3m,nr.employed
count,41188.00000,41188.000000,41188.000000,41188.000000,41188.000000,41188.000000,41188.000000,41188.000000,41188.000000
mean,40.02406,2.567593,962.475454,0.172963,0.081886,93.575664,-40.502600,3.621291,5167.035911
std,10.42125,2.770014,186.910907,0.494901,1.570960,0.578840,4.628198,1.734447,72.251528
min,17.00000,1.000000,0.000000,0.000000,-3.400000,92.201000,-50.800000,0.634000,4963.600000
25%,32.00000,1.000000,999.000000,0.000000,-1.800000,93.075000,-42.700000,1.344000,5099.100000
50%,38.00000,2.000000,999.000000,0.000000,1.100000,93.749000,-41.800000,4.857000,5191.000000
75%,47.00000,3.000000,999.000000,0.000000,1.400000,93.994000,-36.400000,4.961000,5228.100000
max,98.00000,56.000000,999.000000,7.000000,1.400000,94.767000,-26.900000,5.045000,5228.100000


### Nota sobre `pdays`

`pdays` usa el valor centinela `999` para "el cliente nunca fue contactado
antes de esta campaña" (~96.3% de las filas; en `bank-full.csv` el
centinela es `-1`, ver `modeling_standard.ipynb`). No es un dato faltante
real, es una codificación del propio dataset.

In [8]:
print((df["pdays"] == 999).mean())
df.groupby("y")["pdays"].apply(lambda s: (s == 999).mean())

0.9632174419733903


y
no     0.985006
yes    0.791595
Name: pdays, dtype: float64

## 5. Análisis de leakage — por qué se excluye `duration`

In [9]:
# duration = duración de la llamada en segundos. Se conoce SOLO después de
# que la llamada ya terminó -> en producción, al momento de decidir a quién
# llamar, este dato todavía no existe. Usarlo sería data leakage.
print(df.groupby("y")["duration"].mean())
print()
print(df.groupby("y")["duration"].median())

y
no     220.844807
yes    553.191164
Name: duration, dtype: float64

y
no     163.5
yes    449.0
Name: duration, dtype: float64


Como es de esperar, `duration` está fuertemente correlacionada con el
resultado (llamadas más largas -> más probabilidad de "yes"), precisamente
porque es un efecto posterior a la decisión del cliente, no una causa
disponible de antemano. Por eso **se excluye completamente** de
`EXTENDED_FEATURES` y se verifica de forma explícita y programática antes
de entrenar (`src/train_common.py`, `assert "duration" not in features`) y
en cada request de la API (`api/main.py` rechaza el campo `duration` con
HTTP 422).

## 6. Entrenamiento, calibración de umbral y comparación de modelos

El entrenamiento real se ejecuta con:

```
python src/train_extended.py
```

A continuación se reproduce el mismo flujo dentro del notebook, incluyendo
la calibración del umbral de decisión: con ~11% de casos positivos, el
umbral por defecto (0.5) subestima sistemáticamente la clase "yes"; se
busca, sobre un split de **validación** (separado del `X_train`, nunca del
`X_test`), el umbral que maximiza F1 usando la curva precision-recall.

In [10]:
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score, balanced_accuracy_score, roc_auc_score, precision_recall_curve, confusion_matrix
from sklearn.base import clone

from preprocessing import build_pipeline, TARGET

df_clean = df.drop_duplicates()
X = df_clean[EXTENDED_FEATURES]
y = df_clean[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
X_tr2, X_val, y_tr2, y_val = train_test_split(
    X_train, y_train, test_size=0.2, random_state=42, stratify=y_train
)
X_train.shape, X_val.shape, X_test.shape

((32940, 19), (6588, 19), (8236, 19))

In [11]:
lr_pipeline = build_pipeline(
    LogisticRegression(max_iter=1000, class_weight="balanced", random_state=42), variant="extended"
)
lr_grid = GridSearchCV(
    lr_pipeline, param_grid={"model__C": [0.1, 1.0, 3.0]}, scoring="f1_macro", cv=3, n_jobs=-1
)
lr_grid.fit(X_train, y_train)
lr_grid.best_params_

{'model__C': 3.0}

In [12]:
rf_pipeline = build_pipeline(
    RandomForestClassifier(class_weight="balanced", min_samples_leaf=10, random_state=42, n_jobs=-1),
    variant="extended",
)
rf_grid = GridSearchCV(
    rf_pipeline,
    param_grid={"model__n_estimators": [200, 300], "model__max_depth": [8, 10, 12]},
    scoring="f1_macro", cv=3, n_jobs=-1,
)
rf_grid.fit(X_train, y_train)
rf_grid.best_params_

{'model__max_depth': 12, 'model__n_estimators': 300}

In [13]:
def yes_proba(model, X):
    classes = list(model.classes_)
    return model.predict_proba(X)[:, classes.index("yes")]

def tune_threshold(model, X_val, y_val):
    proba = yes_proba(model, X_val)
    precision, recall, thresholds = precision_recall_curve((y_val == "yes").astype(int), proba)
    f1s = 2 * precision * recall / (precision + recall + 1e-9)
    best_idx = int(np.nanargmax(f1s[:-1]))
    return float(thresholds[best_idx])

def evaluate(model, X_test, y_test, threshold):
    proba = yes_proba(model, X_test)
    y_pred = np.where(proba >= threshold, "yes", "no")
    return {
        "f1_score": f1_score(y_test, y_pred, pos_label="yes"),
        "balanced_accuracy": balanced_accuracy_score(y_test, y_pred),
        "roc_auc": roc_auc_score((y_test == "yes").astype(int), proba),
        "confusion_matrix": confusion_matrix(y_test, y_pred, labels=["no", "yes"]),
    }

results = {}
for name, grid in [("logistic_regression", lr_grid), ("random_forest", rf_grid)]:
    val_model = clone(grid.best_estimator_)
    val_model.fit(X_tr2, y_tr2)
    threshold = tune_threshold(val_model, X_val, y_val)

    test_model = clone(grid.best_estimator_)
    test_model.fit(X_train, y_train)
    metrics = evaluate(test_model, X_test, y_test, threshold)
    metrics["decision_threshold"] = threshold
    results[name] = metrics
    print(name, {k: v for k, v in metrics.items() if k != "confusion_matrix"})

logistic_regression {'f1_score': 0.5031166518254675, 'balanced_accuracy': 0.7528992200328407, 'roc_auc': 0.8002552853037767, 'decision_threshold': 0.6365543601104191}


random_forest {'f1_score': 0.5195505617977528, 'balanced_accuracy': 0.7622297482211275, 'roc_auc': 0.8136070030717399, 'decision_threshold': 0.5766380572141523}


## 7. Modelos finales (ambos se despliegan)

Se compara F1 y Balanced Accuracy (con umbral calibrado) en el conjunto de
test (hold-out). El requisito de la Hackathon es entrenar y comparar al
menos 2 modelos -- en vez de descartar el que no gana, **ambos candidatos
se reentrenan con el 100% de los datos disponibles** y se sirven en la API
(`POST /predict/extended` para Random Forest, `POST /predict/extended/logistic_regression`
para Logistic Regression), cada uno con su propio umbral calibrado.

```
model/extended/logistic_regression/model.joblib  + model_info.json
model/extended/random_forest/model.joblib        + model_info.json
model/extended/comparison.json                    (metricas de ambos + cual se recomendaria)
```

In [14]:
import json

with open("../model/extended/comparison.json") as f:
    comparison = json.load(f)

print("Dataset:", comparison["dataset"])
print("Modelo recomendado (si solo se desplegara uno):", comparison["recommended_model"])
print()
for model_key, metrics in comparison["all_candidates"].items():
    print(f"--- {model_key} ---")
    for k, v in metrics.items():
        if k not in ("confusion_matrix", "confusion_matrix_labels"):
            print(f"  {k}: {v:.4f}" if isinstance(v, float) else f"  {k}: {v}")

Dataset: extended (bank-additional-full.csv, 20 variables, bonus)
Modelo recomendado (si solo se desplegara uno): random_forest

--- logistic_regression ---
  decision_threshold: 0.6366
  f1_score: 0.5031
  balanced_accuracy: 0.7529
  precision: 0.4287
  recall: 0.6088
  roc_auc: 0.8003
--- random_forest ---
  decision_threshold: 0.5766
  f1_score: 0.5196
  balanced_accuracy: 0.7622
  precision: 0.4456
  recall: 0.6228
  roc_auc: 0.8136


## 8. Comparación con el dataset estándar (17 variables)

Este modelo (20 variables) obtiene mejores métricas que el modelo entrenado
con `bank-full.csv` (17 variables, ver `modeling_standard.ipynb`). La
diferencia proviene de los indicadores macroeconómicos adicionales
(`emp.var.rate`, `cons.price.idx`, `cons.conf.idx`, `euribor3m`,
`nr.employed`), que aportan señal relevante sobre el contexto de cada
campaña y no están disponibles en el dataset estándar (ver comparación
completa en el README, sección 5).

## 9. Verificación explícita: `duration` fuera del modelo final

In [15]:
assert "duration" not in EXTENDED_FEATURES
print("OK: duration NOT IN final_features")

OK: duration NOT IN final_features
